# [16.8] Do SHAPley and Mechanistic Interpretability Agree?

This notebook asks one bounded question: when can SHAPley-style attribution and a mechanistic ground-truth score be said to agree?

You will not answer this with a pretty heatmap. You will answer it with finite games, exact tests, deletion/insertion consequences, and a CUDA model organism with a shuffled-label control.

The key lesson is that disagreement is often about the **player set**. A single-feature Shapley value can miss an XOR mechanism because the real player is a pair.

## Core Question

When attribution and mechanistic scores disagree, are the scores wrong, or are they scoring different player sets?


## Learning Objectives

By the end, you should be able to:

1. Rank attribution scores deterministically.
2. Convert a ranking into deletion and insertion consequence curves.
3. Check additive agreement against a known mechanistic score.
4. Diagnose an XOR disagreement where interactions are the correct player set.
5. Write an agreement matrix and visible artifact bundle.
6. Interpret the CUDA report without making broad large-model claims.

<details>
<summary>Help - what counts as agreement?</summary>

Agreement means the top players and consequences line up: high rank correlation, exact top-k overlap, top deletion beats a non-top baseline, and the correct interaction pair is recovered.

</details>


In [ ]:
import csv
import json
import sys
from pathlib import Path

import torch as t

chapter = "chapter16_shapley_attribution_baselines"
section = "part8_shapley_mechinterp_agreement"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part8_shapley_mechinterp_agreement.tests as tests
import part8_shapley_mechinterp_agreement.utils as utils

from arena_ext.shapley_attribution import (
    additive_game,
    attribution_agreement_report,
    interaction_agreement_report,
    pairwise_shapley_interactions,
    topk_overlap_fraction,
    xor_game,
)
from arena_ext.shapley_neural_game import (
    NEURAL_GAME_NUM_PLAYERS,
    coalition_table_from_true_game,
)

MAIN = True
ARTIFACT_DIR = section_dir / "artifacts"


## Exercise 1 - rank attribution scores

> ```yaml
> Difficulty: medium
> Importance: high
> ```


Implement a deterministic descending rank helper.

In [ ]:
def _rank_desc(scores: t.Tensor) -> list[int]:
    """
    Return indices sorted by descending score.

    Inputs:
        scores: tensor of shape [num_players].
    Returns:
        Python list of player indices, highest score first.
    """
    # EXERCISE
    # YOUR CODE HERE
    raise NotImplementedError()
    # END EXERCISE


if MAIN:
    tests.test_rank_desc_toy_oracle(_rank_desc)


<details>
<summary>Expected output</summary>

`test_rank_desc_toy_oracle` should pass. The tie case keeps a deterministic order.

</details>

<details>
<summary>Solution</summary>

Try the exercise first, then compare against the paired solutions notebook.

</details>


## Exercise 2 - analytic mechanism scores

> ```yaml
> Difficulty: medium
> Importance: high
> ```


Compute feature scores from the generated rule decomposition, not from model-output Shapley.

In [ ]:
def analytic_neural_game_mechanistic_scores() -> t.Tensor:
    """
    Return feature scores from the generated rule decomposition.

    Rule:
        0.25 + 1.2*x0 - 0.7*x1 + 1.6*x2 + 0.9*x3
        + 2.2*x0*x2 - 1.5*x1*x3
    Allocate each pair interaction equally to its two features.
    """
    # EXERCISE
    # YOUR CODE HERE
    raise NotImplementedError()
    # END EXERCISE


if MAIN:
    tests.test_analytic_neural_game_mechanistic_scores_toy_oracle(
        analytic_neural_game_mechanistic_scores
    )


<details>
<summary>Expected output</summary>

The analytic scores should be `[2.3, -1.45, 2.7, 0.15]`.

</details>

<details>
<summary>Solution</summary>

Try the exercise first, then compare against the paired solutions notebook.

</details>


## Exercise 3 - consequence curves

> ```yaml
> Difficulty: medium
> Importance: high
> ```


Turn rankings into deletion and insertion curves.

In [ ]:
def _curve_from_rank(values: dict[frozenset[int], float], rank: list[int], mode: str) -> list[dict]:
    """
    Build deletion or insertion curve points from a ranking.

    Inputs:
        values: complete coalition value table.
        rank: player indices sorted by decreasing importance.
        mode: either 'deletion' or 'insertion'.
    Returns:
        List of dictionaries with step, player, and value fields.
    """
    # EXERCISE
    # YOUR CODE HERE
    raise NotImplementedError()
    # END EXERCISE


if MAIN:
    tests.test_curve_from_rank_deletion_and_insertion(_curve_from_rank)


<details>
<summary>Expected output</summary>

Deletion starts from the full coalition; insertion starts from the empty coalition.

</details>

<details>
<summary>Solution</summary>

Try the exercise first, then compare against the paired solutions notebook.

</details>


## Exercise 4 - additive agreement positive control

> ```yaml
> Difficulty: medium
> Importance: high
> ```


In [ ]:
def _tensor_report(report) -> dict:
    result = report.__dict__.copy()
    for key, value in list(result.items()):
        if hasattr(value, "tolist"):
            result[key] = value.tolist()
    return result


def additive_agreement_smoke_test() -> dict:
    """Return rank, top-k, and deletion metrics for the additive positive control."""
    # EXERCISE
    # YOUR CODE HERE
    raise NotImplementedError()
    # END EXERCISE


if MAIN:
    tests.test_additive_agreement_smoke_test(additive_agreement_smoke_test)


<details>
<summary>Expected output</summary>

`topk_overlap == 1.0`, `spearman_correlation > 0.99`, and top deletion beats baseline.

</details>

<details>
<summary>Solution</summary>

Try the exercise first, then compare against the paired solutions notebook.

</details>


## Exercise 5 - XOR disagreement control

> ```yaml
> Difficulty: medium
> Importance: high
> ```


In [ ]:
def xor_disagreement_smoke_test() -> dict:
    """Return ordinary Shapley and pair-interaction metrics for the XOR control."""
    # EXERCISE
    # YOUR CODE HERE
    raise NotImplementedError()
    # END EXERCISE


if MAIN:
    tests.test_xor_disagreement_smoke_test(xor_disagreement_smoke_test)


<details>
<summary>Expected output</summary>

Ordinary Shapley has zero single-feature value, while pair interaction recovers value `2.0`.

</details>

<details>
<summary>Solution</summary>

Try the exercise first, then compare against the paired solutions notebook.

</details>


## Exercise 6 - agreement artifact bundle

> ```yaml
> Difficulty: hard
> Importance: high
> ```


In [ ]:
def _artifact_display_path(path: Path) -> str:
    try:
        return str(path.relative_to(root_dir))
    except ValueError:
        return str(path)


def write_agreement_artifacts(
    *,
    output_dir: Path,
    model_values: dict[frozenset[int], float],
    true_values: dict[frozenset[int], float],
    shuffled_values: dict[frozenset[int], float],
    agreement,
    shuffled_agreement,
    model_interactions: t.Tensor,
    true_interactions: t.Tensor,
) -> dict:
    """Write the matrix, consequence plots, heatmap, and disagreement notes."""
    # EXERCISE
    # YOUR CODE HERE
    raise NotImplementedError()
    # END EXERCISE


if MAIN:
    tests.test_write_agreement_artifacts_contract()


<details>
<summary>Expected output</summary>

The artifact writer creates a seven-row matrix, two curve PNGs, a heatmap PNG, and a Markdown disagreement note.

</details>

<details>
<summary>Solution</summary>

Try the exercise first, then compare against the paired solutions notebook.

</details>


## Exercise 7 - notebook contract

> ```yaml
> Difficulty: medium
> Importance: medium
> ```


In [ ]:
def run_smoke_test(cpu: bool = True) -> dict:
    _ = cpu
    # EXERCISE
    # YOUR CODE HERE
    raise NotImplementedError()
    # END EXERCISE


if MAIN:
    tests.test_notebook_contract(run_smoke_test)


## Exercise 8 - inspect committed CUDA report

> ```yaml
> Difficulty: medium
> Importance: high
> ```


In [ ]:
def run_gpu_test(max_vram_gb: float = 24.0) -> dict:
    """Run the real CUDA finite-game agreement preflight for this section."""
    from chapter16_shapley_attribution_baselines.exercises.part8_shapley_mechinterp_agreement import solutions as reference_solutions

    return reference_solutions.run_gpu_test(max_vram_gb=max_vram_gb)


def run_full_experiment(max_vram_gb: float = 24.0) -> dict:
    """Run the validated experiment path used by verification_report.json."""
    return run_gpu_test(max_vram_gb=max_vram_gb)


In [ ]:
report_path = section_dir / "verification_report.json"
report = json.loads(report_path.read_text())
gpu = report["metrics"]["gpu_test"]

summary = {
    "preflight_passed": gpu["preflight_passed"],
    "model_family": gpu["model_family"],
    "spearman_correlation": gpu["spearman_correlation"],
    "topk_overlap": gpu["topk_overlap"],
    "deletion_gap": gpu["deletion_drop"] - gpu["random_baseline_drop"],
    "interaction_max_abs_error": gpu["interaction_max_abs_error"],
    "shuffled_control_rejected": gpu["shuffled_control_rejected"],
    "peak_vram_gb": gpu["peak_vram_gb"],
}
utils.print_report("Committed CUDA agreement report", summary)

if MAIN:
    tests.test_committed_gpu_report_records_agreement_and_controls()


## Signature Result

The committed CUDA report should show:

| Case | Metric | Expected |
| --- | ---: | ---: |
| trained finite game | Spearman | about 1.0 |
| trained finite game | top-k overlap | 1.0 |
| top deletion | beats non-top baseline | true |
| interaction recovery | max error | less than 1e-4 |
| shuffled control | rejected | true |

<details>
<summary>Interpreting the result</summary>

The positive result is bounded: exact model-output Shapley agrees with analytic mechanism scores on a finite generated rule.
The negative control matters because a trained model can fit shuffled targets while failing the mechanism agreement test.

</details>


## Limitations and Bonus Anomaly Hunting

This notebook does not prove method agreement on arbitrary transformer circuits. It proves a finite protocol: define the player set, test consequences, recover interactions, and reject shuffled controls.

Bonus ideas:

1. Add a three-way interaction where pair interactions fail.
2. Swap the player set from features to edges and see which agreement rows survive.
3. Repeat the CUDA finite game across seeds and look for spurious shuffled alignment.
